<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/04_simple_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04_simple_CNN

## Build a simple CNN

This tutorial shows you how to build a simple CNN for the classification of

nucleotide sequences.

First, you will define functions to build and compile your neural network.

Second, you will define functions to convert sequences into one hot encoding to

create a dataset that can be used by your network.

Finally, you will train and test your network on a dummy dataset (or your 

personal file if you have it).



### import required modules
The following cells will install TensorFlow 2.0 on the VM

In [0]:
import tensorflow as tf
from tensorflow import keras as K

In [0]:
if tf.__version__ != "2.0.0":
  !pip install --upgrade tensorflow

## Restart the RunTime if you just installed TF2 !

Go to the menu bar > Runtime > Restart runtime...

Done, bravo!


### Here additional code to mount your Google Drive folder on the VM

You can skip this part if you are not interested.

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### create network architecture
This section defines the network architecture, a simple one. 

It is made by three 1D Convolutional Networks, followed by an artificial neural 

network made of three dense layer of 512 neurons each.

You can update, modify or add new layers. You can look at the Keras 

documentation at https://keras.io/ or read the book 'Deep Learning with Python'

written by F. Chollet (you can also follow him on Twitter).


In [0]:
def create_architecture(
    sequence_size = 50,
    channel = 6,
    classes = 2
    ):
  '''
  fun create the model based on sequence input
  size.

  paramenters:
  sequence_size=length of sequence in nt
  channel=each channel corresponds to a nucleotide.
  classes=number of output classes
  '''
  # create simple Keras model
  model = K.models.Sequential()

  # add first conv layer  
  model.add(K.layers.Conv1D(
      filters = 128,
      kernel_size = (6),
      padding='same',
      data_format="channels_last",
      activation='relu',
      input_shape=(sequence_size,channel)))

  model.add(K.layers.BatchNormalization() )
  model.add(K.layers.MaxPooling1D())
  model.add(K.layers.Dropout(0.2))

  # add first conv layer  
  model.add(K.layers.Conv1D(
    filters = 128,
    kernel_size = (6),
    padding='same',
    data_format="channels_last",
    activation='relu'))

  model.add(K.layers.BatchNormalization() )
  model.add(K.layers.MaxPooling1D())
  model.add(K.layers.Dropout(0.2))

  # add first conv layer  
  model.add(K.layers.Conv1D(
    filters = 128,
    kernel_size = (6),
    padding='same',
    data_format="channels_last",
    activation='relu')
  )

  model.add(K.layers.BatchNormalization() )
  model.add(K.layers.MaxPooling1D())
  model.add(K.layers.Dropout(0.2))

  model.add(K.layers.Flatten())

  model.add(K.layers.Dense( 512, activation = "relu" ))
  model.add(K.layers.BatchNormalization())
  model.add(K.layers.Dropout(0.2))

  model.add(K.layers.Dense( 512, activation = "relu" ))
  model.add(K.layers.BatchNormalization())
  model.add(K.layers.Dropout(0.2))

  model.add(K.layers.Dense( 512, activation = "relu" ))
  model.add(K.layers.BatchNormalization())
  model.add(K.layers.Dropout(0.2))

  model.add(K.layers.Dense(classes, activation = "softmax"))

  return(model)

### Compile Network and add optimizer

There are two common optimizer for CNN, one is Stochastic gradient descent 

optimizer (SGD) and the second one is ADAM. You can read about them at the 

following URL: https://keras.io/optimizers/#sgd .

The Learning rate paramenters is the main player that affects the  model

performances in therms of accuracy and precision (as well as training speed).

ML and DL are emphirical science, therefore you will need to try different 

settings to train a good model (this settings are called hyperparamenters).

Another important setting is the loss function; I left categorical crossentropy

since it allows to distinguish between 2 or more classess. You can read more 

guess where ... https://keras.io/losses/#usage-of-loss-functions


In [0]:
def compile_network(
    model,
    optimizer='adam'
):
  if optimizer=='sgd':    
    sgd = K.optimizers.SGD(learning_rate=0.001, momentum=0.0, nesterov=True)
  elif optimizer=='adam':
    adam = K.optimizers.Adam(
      learning_rate=0.01, beta_1=0.9, beta_2=0.999, amsgrad=False
      )
  else:
    print('unknown optimizer')
    return None

  model.compile(
      optimizer=adam,
      loss='categorical_crossentropy',
      metrics=['accuracy'],
      
  )
  return model

## you model is ready, now we need a dataset to train it.

we can first import the required python modules.

In [0]:
import numpy as np
import pandas as pd

### Dataset to One Hot Encoding

This is a very simple idea of how to convert you nucleotide input sequences to 

one hot encoding. These cells will convert your input dataset into a one hot 

encoding numpy array that can feed your network.



In [0]:
def sequence_to_ohe(
    dataset=None,
    sequence_size=50,
    channel=6
    ):
  '''
  fun builds the ohe array of each 
  sample sequence.

  paramenters:
  sequence_size= can corresponds to the
  length of the input sequences
  (if all the same) or you can define an
  arbitrary number.
  channel=the channel size, which correspond
  to the number of possible nucleotides.

  '''

  def ohe_categories(nucleotide):
    '''
    fun defines the position of each
    nucleotide inside the ohe numpy
    array. I choose to have 6 possible 
    nucleotide in order to handle both
    transcriptome sequence as well as
    Ns inside the sequence. It returns
    the position in the channel for the
    specific input nucleotide.

    parameters:
    nucleotide= a single nucleotide letter
    '''
    channel = {
        'A' : 0,
        'T' : 1,
        'C' : 2,
        'G' : 3,
        'U' : 4,
        'N' : 5
    }
    return channel[nucleotide]

  samples_size = len(dataset)
  ohe_dataset = np.zeros((samples_size, sequence_size, channel))

  for index, sequence in enumerate(dataset):
    for pos, nucleotide in enumerate(sequence):
      ohe_dataset[index, pos, ohe_categories(nucleotide)] = 1 
  
  return ohe_dataset


### load file with dataset

I designed the function in order to handle a tab-delimited input file with 

the following columns: sequence and label (and no header!).

e.g. training_dataset.tsv

CGTCGTCG...    positive  
CGACGACG...    positive  
CGTGGACG...    positive  
AAAAAAAA...    negative  


The number of classes will be defined after reading the input dataset.

### create dummy dataset

This code will generate a dummy dataset to show and test the current code


In [0]:
import io
import random

In [0]:
def make_dummy(
    sequence_size=50,
    db_size=1000
    ):
  random_db = str()
  count = 0
  while count != db_size:
    random_number = random.randint(1,25)
    random_label = random.choice(['positive', 'negative'])
    if random_label == 'positive':
      random_seq = f'{ "C" * random_number}{"G" * (sequence_size - random_number)}'
    else:
      random_seq = f'{ "A" * random_number}{"T" * (sequence_size - random_number)}'
    
    random_db += f'{random_seq}\t{random_label}\n'
    count += 1
  bed_file = io.StringIO(random_db)
  return bed_file

In [0]:
dummy_db = make_dummy()
df = pd.read_csv(dummy_db, sep='\t', names=['sequence', 'label'])

### Load your dataset
You have a personal dataset that satisfied the requirement of the 

script, thus a tab-separated text file with no header where the

firt and second columns are: sequence and label. You can use 

the pandas read_csv to load your table by specifying the

file name. You can load your file on your Google Drive and 

then link it to the VM (see above instructions).

In [0]:
your_file_name = ''

if your_file_name:
  df = pd.read_csv(your_file_name, sep='\t', names=['sequence', 'label'])

### Prepare Dataset for training

We are now converting you df to one hot encoding.

take advantages of pandas method 'get_dummies' to convert our labels to
one hot encoding. We also split it into two new dataframe, labels and 
sequences (y and x).


In [0]:
sequence_size = 50

In [14]:
sequence_df = sequence_to_ohe(
    dataset=df['sequence'].tolist(),
    sequence_size=sequence_size,
)

labels_df = pd.get_dummies(df['label'])


if sequence_df.shape[0] == labels_df.shape[0]:
  print('dataset OK')
else:
  print('sequence and label shapes are different, something went wrong...')

print(
    'sequence_df samples',
    sequence_df.shape,
    'labels_df samples',
    labels_df.shape,
    sep='\t'
    )


dataset OK
sequence_df samples	(1000, 50, 6)	labels_df samples	(1000, 2)


### Train and test splitting

Since a proper pipeline for the development of a model required both training

and test dataset, we will split your dataset by using sklearn package and its

module train_test_split. Amazing. Really .. it is amazing!

Since it is amazing, you can read more here:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html



In general, X identified samples while y identifies labels.

In general, it is always raccomended to define a random state for the 

splitting, so then your results can be reproducibile in the future.

Random state means that you will always get the same random sampling.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
          sequence_df, labels_df, test_size=0.33, random_state=1989)

print(
    'X_train sequences',
    X_train.shape[0],
    'y_train labels',
    y_train.shape[0],
    sep='\t'
    )

print(
    'X_test sequences',
    X_test.shape[0],
    'y_test labels',
    y_test.shape[0],
    sep='\t'
    )


X_train sequences	670	y_train labels	670
X_test sequences	330	y_test labels	330


### create the network

In [0]:
classes = labels_df.shape[-1]


model = create_architecture(sequence_size=sequence_size, channel=6, classes=classes)
model = compile_network( model=model, optimizer='adam')

### training network

We fit the network with our train dataset using starndard paramenters;

you can read more at Keras.io: https://keras.io/models/sequential/#fit;

history will save model performances as an attribute. After training 

you can look at it by typing history.history;

otherwise, you can look at additional usefull keras callbacks like

csv_log or early_stop. Look at: https://keras.io/callbacks/


In [17]:
history = model.fit(
    X_train,
    y_train.to_numpy(),
    batch_size=32,
    epochs=10,
    validation_split=0.2
)

Train on 536 samples, validate on 134 samples
Epoch 1/10
536/536 [==============================] - 4s 7ms/sample - loss: 0.0887 - accuracy: 0.9627 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/10
536/536 [==============================] - 1s 2ms/sample - loss: 4.0282e-05 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 3/10
536/536 [==============================] - 1s 2ms/sample - loss: 0.0024 - accuracy: 0.9981 - val_loss: 6.8375e-06 - val_accuracy: 1.0000
Epoch 4/10
536/536 [==============================] - 1s 2ms/sample - loss: 2.3944e-04 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 5/10
536/536 [==============================] - 1s 2ms/sample - loss: 1.1981e-04 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 6/10
536/536 [==============================] - 1s 2ms/sample - loss: 1.3115e-06 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 7/10
536/536 [==============================

### evaluate your model performance on an indipendent dataset

We created the indipendent dataset as test set, and we can now understand if

our trained model is any good in discerning our classes. In our case, the output

metrics will be loss and accuracy. we want loss to be as smaller as possible and

accuracy closer to 1.

In [18]:
metrics = model.evaluate(
    X_test,
    y_test.to_numpy(),
    verbose=0
)

print('model evaluation on unknown dataset [loss, accuracy]:', metrics)

model evaluation on unknown dataset [loss, accuracy]: [0.0, 1.0]
